This notebook preprocesses two data sources for the electricity-demand forecasting project:

1. **SvK electricity data**  
   Raw Svenska kraftnät Excel files are read, separated into consumption, generation, and losses, checked for data quality, and saved as clean CSV files while preserving regions, categories, signs, MWh units, and timestamps.

2. **ERA5-Land weather data**  
   Weather data are read from the downloaded archive, merged by timestamp and location, and saved as a clean CSV file. The data include temperature, dew point, wind, precipitation, and solar radiation for Stockholm as a representative location for the SE3 bidding zone.

The processed files are stored in `data/clean/SvK` and `data/clean/weather`.

# 01 SvK Energy Consumption and Generation Data Analysis

Explore hourly electricity consumption, generation, and losses from **data/SvK** for 2018-2026. Regions and categories remain separate columns, and temporal summaries are calculated independently for each column without combining regions or categories.

Run from the repository root with the `.venv-win` kernel. The preprocessing preserves source signs and MWh units. Timestamps retain the source convention, so confirm timezone and daylight-saving treatment before interpreting local-hour patterns. The 2026 file covers only January-June, so annual comparisons should use matching periods.

In [12]:
from pathlib import Path
import re
import warnings

import pandas as pd
import numpy as np

import cdsapi
from pathlib import Path

from pathlib import Path
from zipfile import ZipFile

from functools import reduce

In [5]:
ROOT = Path.cwd()
INPUT_DIR = ROOT / "data" / "SvK"
OUTPUT_DIR = ROOT / "data" / "clean" / "SvK"

MAIN = {
    "Timmätt förbr": "metered_consumption",
    "Avkopplingsb.": "interruptible",
    "Avkopplingsbar": "interruptible",
    "Energilager": "energy_storage",
    "Timmätta": "metered",
    "Ospec.": "unspecified",
    "Vattenkraft": "hydro",
    "Vindkraft": "wind",
    "Kärnkraft": "nuclear",
    "Värmekraft": "thermal",
    "Solkraft": "solar",
    "Schablonleverans": "profiled_supply",
}

SUB = {
    "exkl. avk.last": "excluding_interruptible_load",
    ">50 MW": "over_50mw",
    "last": "load",
    "förbrukning": "consumption",
    "förluster": "losses",
    "produktion": "generation",
    "landbaserad": "onshore",
    "havsbaserad": "offshore",
}


def read_svk_file(path):
    raw = pd.read_excel(path, sheet_name=0, header=None, engine="xlrd")
    body = raw.iloc[4:].copy()

    timestamps = pd.to_datetime(
        body.iloc[:, 0],
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )

    valid = timestamps.notna()

    if not valid.any():
        raise ValueError(f"{path.name}: no timestamp rows found")

    index = pd.DatetimeIndex(
        timestamps[valid],
        name="datetime",
    )

    groups = {
        "consumption": {},
        "generation": {},
        "losses": {},
    }

    for col in range(1, raw.shape[1]):
        if raw.iloc[:, col].isna().all():
            continue

        main, sub, zone, unit = (
            "" if pd.isna(value) else str(value).strip()
            for value in raw.iloc[:4, col]
        )

        if not any((main, sub, zone, unit)):
            warnings.warn(
                f"{path.name}: unlabeled column {col + 1} excluded"
            )
            continue

        if main not in MAIN:
            raise ValueError(f"{path.name}: unknown main category {main!r}")

        if sub not in SUB:
            raise ValueError(f"{path.name}: unknown subcategory {sub!r}")

        if zone not in {"SE1", "SE2", "SE3", "SE4"}:
            raise ValueError(f"{path.name}: unknown region {zone!r}")

        if unit != "MWh":
            raise ValueError(f"{path.name}: unexpected unit {unit!r}")

        if sub == "förluster":
            group = "losses"
        elif (
            sub == "produktion"
            or (
                main == "Vindkraft"
                and sub in {"landbaserad", "havsbaserad"}
            )
        ):
            group = "generation"
        elif (
            main in {
                "Timmätt förbr",
                "Avkopplingsb.",
                "Avkopplingsbar",
            }
            or sub == "förbrukning"
        ):
            group = "consumption"
        else:
            raise ValueError(
                f"{path.name}: cannot classify {(main, sub)}"
            )

        column_name = "_".join(
            [zone.lower(), MAIN[main], SUB[sub]]
        )

        if column_name in groups[group]:
            raise ValueError(
                f"{path.name}: duplicate column {column_name}"
            )

        source_values = body.loc[valid, col]
        numeric_values = pd.to_numeric(
            source_values,
            errors="coerce",
        )

        invalid = source_values.notna() & numeric_values.isna()

        if invalid.any():
            warnings.warn(
                f"{path.name}: {invalid.sum()} invalid values in "
                f"{column_name} became NaN"
            )

        groups[group][column_name] = numeric_values.to_numpy()

    return {
        group: pd.DataFrame(
            columns,
            index=index,
        ).sort_index(kind="stable")
        for group, columns in groups.items()
    }


# Find one Excel file for each year
files = {}

for path in sorted(INPUT_DIR.glob("timvarden-*.xls")):
    match = re.match(r"timvarden-(\d{4})", path.name)

    if match:
        year = int(match.group(1))

        if 2018 <= year <= 2026:
            if year in files:
                raise ValueError(
                    f"Multiple files found for {year}"
                )

            files[year] = path

missing_years = sorted(
    set(range(2018, 2027)) - files.keys()
)

if missing_years:
    raise FileNotFoundError(
        f"Missing files for years: {missing_years}"
    )


# Read and combine all years
collected = {
    "consumption": [],
    "generation": [],
    "losses": [],
}

for year, path in sorted(files.items()):
    tables = read_svk_file(path)

    for group, frame in tables.items():
        if not (frame.index.year == year).all():
            raise ValueError(
                f"{path.name}: timestamps outside year {year}"
            )

        collected[group].append(frame)

    print(f"{path.name}: read successfully")


# Create final dataframes
tables = {}

for group, frames in collected.items():
    frame = pd.concat(
        frames,
        axis=0,
        join="outer",
        sort=False,
    ).sort_index(kind="stable")

    if frame.index.has_duplicates:
        warnings.warn(
            f"{group}: duplicate timestamps were preserved"
        )

    tables[group] = frame


df_consumption = tables["consumption"]
df_generation = tables["generation"]
df_losses = tables["losses"]


# Export the cleaned files
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, frame in {
    "consumption": df_consumption,
    "generation": df_generation,
    "losses": df_losses,
}.items():
    output_file = OUTPUT_DIR / f"{name}_2018_2026.csv"
    frame.to_csv(output_file)

    print(
        f"{name}: {frame.shape}, "
        f"{frame.index.min()} through {frame.index.max()}"
    )
    print(f"Saved to: {output_file}")

C:\Users\wuton\AppData\Local\Temp\ipykernel_33604\3405402035.py:69: UserWarning: timvarden-2018-01-12.xls: unlabeled column 40 excluded
  warnings.warn(
C:\Users\wuton\AppData\Local\Temp\ipykernel_33604\3405402035.py:69: UserWarning: timvarden-2018-01-12.xls: unlabeled column 41 excluded
  warnings.warn(


timvarden-2018-01-12.xls: read successfully
timvarden-2019-01-12.xls: read successfully
timvarden-2020-01-12.xls: read successfully
timvarden-2021-01-12.xls: read successfully
timvarden-2022-01-12.xls: read successfully
timvarden-2023-01-12.xls: read successfully
timvarden-2024-01-12_.xls: read successfully
timvarden-2025-01-12.xls: read successfully
timvarden-2026-01-06.xls: read successfully
consumption: (74472, 20), 2018-01-01 00:00:00 through 2026-06-30 23:00:00
Saved to: d:\coding_study\repo\electricity_demand_forecasting\data\clean\SvK\consumption_2018_2026.csv
generation: (74472, 33), 2018-01-01 00:00:00 through 2026-06-30 23:00:00
Saved to: d:\coding_study\repo\electricity_demand_forecasting\data\clean\SvK\generation_2018_2026.csv
losses: (74472, 8), 2018-01-01 00:00:00 through 2026-06-30 23:00:00
Saved to: d:\coding_study\repo\electricity_demand_forecasting\data\clean\SvK\losses_2018_2026.csv


In [6]:
reference_index = df_consumption.index

overview = pd.DataFrame([
    {'Dataset': name, 'Rows': len(frame), 'Columns': frame.shape[1],
     'Start': frame.index.min(), 'End': frame.index.max(),
     'Granularity': frame.index.to_series().diff().dropna().mode().iloc[0],
     'Duplicate timestamps': frame.index.duplicated().sum(),
     'Missing hourly timestamps': len(
         pd.date_range(frame.index.min(), frame.index.max(), freq='h').difference(frame.index)
     ),
     'Same timestamps as Consumption': frame.index.equals(reference_index),
     'Sorted': frame.index.is_monotonic_increasing}
    for name, frame in tables.items()
]).set_index('Dataset')
display(overview)

for name, frame in tables.items():
    print(name)
    display(frame.head())
    display(pd.DataFrame({
        'Missing (%)': frame.isna().mean().mul(100),
        'Zero (%)': frame.eq(0).sum().div(frame.count().replace(0, np.nan)).mul(100),
        'First valid': frame.apply(lambda s: s.first_valid_index()),
        'Last valid': frame.apply(lambda s: s.last_valid_index()),
    }))

,Rows,Columns,Start,End,Granularity,Duplicate timestamps,Missing hourly timestamps,Same timestamps as Consumption,Sorted
Dataset,,,,,,,,,
consumption,74472,20,2018-01-01,2026-06-30 23:00:00,0 days 01:00:00,0,0,True,True
generation,74472,33,2018-01-01,2026-06-30 23:00:00,0 days 01:00:00,0,0,True,True
losses,74472,8,2018-01-01,2026-06-30 23:00:00,0 days 01:00:00,0,0,True,True


consumption


,se1_metered_consumption_excluding_interruptible_load,se2_metered_consumption_excluding_interruptible_load,se3_metered_consumption_excluding_interruptible_load,se4_metered_consumption_excluding_interruptible_load,se1_interruptible_load,se2_interruptible_load,se3_interruptible_load,se4_interruptible_load,se1_profiled_supply_consumption,se2_profiled_supply_consumption,se3_profiled_supply_consumption,se4_profiled_supply_consumption,se1_metered_consumption_over_50mw,se2_metered_consumption_over_50mw,se3_metered_consumption_over_50mw,se4_metered_consumption_over_50mw,se1_energy_storage_consumption,se2_energy_storage_consumption,se3_energy_storage_consumption,se4_energy_storage_consumption
datetime,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,-879.546411,-1324.912999,-5082.332339,-1032.212224,-15.139255,-4.28215,-35.379498,-1.820,-320.852311,-677.512910,-4146.389119,-1315.833745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 01:00:00,-854.412614,-1301.169237,-5068.952361,-1021.143207,-8.588081,-4.34295,-33.090146,-2.034,-312.872325,-655.939945,-3985.215149,-1266.761195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 02:00:00,-849.282198,-1263.349477,-5045.623694,-1002.607815,-15.368199,-4.29760,-32.917950,-2.210,-304.099395,-635.338786,-3840.439564,-1214.776332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 03:00:00,-855.437041,-1261.090204,-5018.433271,-1009.783807,-10.771903,-4.32470,-32.660058,-1.980,-296.183806,-619.271420,-3693.665116,-1162.921100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 04:00:00,-864.749933,-1257.587183,-5010.117105,-1034.710987,-15.183108,-4.49015,-32.197542,-2.072,-293.452582,-613.083015,-3609.149193,-1136.420289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Missing (%),Zero (%),First valid,Last valid
se1_metered_consumption_excluding_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se2_metered_consumption_excluding_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se3_metered_consumption_excluding_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se4_metered_consumption_excluding_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se1_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se2_interruptible_load,0.000000,0.028199,2018-01-01,2026-06-30 23:00:00
se3_interruptible_load,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se4_interruptible_load,0.000000,21.406703,2018-01-01,2026-06-30 23:00:00
se1_profiled_supply_consumption,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se2_profiled_supply_consumption,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00


generation


,se1_unspecified_generation,se2_unspecified_generation,se3_unspecified_generation,se4_unspecified_generation,se1_hydro_generation,se2_hydro_generation,se3_hydro_generation,se4_hydro_generation,se1_wind_generation,se2_wind_generation,...,se3_energy_storage_generation,se4_energy_storage_generation,se1_wind_onshore,se2_wind_onshore,se3_wind_onshore,se4_wind_onshore,se1_wind_offshore,se2_wind_offshore,se3_wind_offshore,se4_wind_offshore
datetime,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,0.0000,0.01062,11.56062,12.782519,1567.682081,2452.401968,1371.185247,283.274302,248.367512,466.054477,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 01:00:00,0.5540,0.00833,11.89710,12.719574,1168.017709,2149.582161,1253.252445,283.654948,238.104265,449.537397,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 02:00:00,0.7710,0.01621,11.62666,12.306289,985.827853,2004.395075,1216.744223,283.619570,228.781771,497.112675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 03:00:00,1.0132,0.02070,11.88815,12.639589,973.166394,1905.183572,1200.083650,283.563376,218.608491,588.583537,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 04:00:00,0.7650,0.00064,11.62433,12.972690,1018.671665,1987.470981,1210.767642,283.555877,187.179221,639.385540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Missing (%),Zero (%),First valid,Last valid
se1_unspecified_generation,0.000000,43.226985,2018-01-01,2026-06-30 23:00:00
se2_unspecified_generation,0.000000,2.502954,2018-01-01,2026-06-30 23:00:00
se3_unspecified_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se4_unspecified_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se1_hydro_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se2_hydro_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se3_hydro_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se4_hydro_generation,0.000000,0.000000,2018-01-01,2026-06-30 23:00:00
se1_wind_generation,5.833065,0.000000,2018-01-01,2025-12-31 23:00:00
se2_wind_generation,5.833065,0.000000,2018-01-01,2025-12-31 23:00:00


losses


,se1_profiled_supply_losses,se2_profiled_supply_losses,se3_profiled_supply_losses,se4_profiled_supply_losses,se1_metered_losses,se2_metered_losses,se3_metered_losses,se4_metered_losses
datetime,,,,,,,,
2018-01-01 00:00:00,-28.075403,-52.288821,-312.810246,-94.713027,NaN,NaN,NaN,NaN
2018-01-01 01:00:00,-27.464304,-50.574203,-299.410340,-91.175143,NaN,NaN,NaN,NaN
2018-01-01 02:00:00,-26.685033,-48.943735,-288.456633,-87.410734,NaN,NaN,NaN,NaN
2018-01-01 03:00:00,-25.991393,-47.682140,-277.178742,-83.639353,NaN,NaN,NaN,NaN
2018-01-01 04:00:00,-25.749979,-47.189229,-270.630666,-81.697897,NaN,NaN,NaN,NaN


,Missing (%),Zero (%),First valid,Last valid
se1_profiled_supply_losses,0.00000,0.0,2018-01-01,2026-06-30 23:00:00
se2_profiled_supply_losses,0.00000,0.0,2018-01-01,2026-06-30 23:00:00
se3_profiled_supply_losses,0.00000,0.0,2018-01-01,2026-06-30 23:00:00
se4_profiled_supply_losses,0.00000,0.0,2018-01-01,2026-06-30 23:00:00
se1_metered_losses,11.76281,0.0,2019-01-01,2026-06-30 23:00:00
se2_metered_losses,11.76281,0.0,2019-01-01,2026-06-30 23:00:00
se3_metered_losses,11.76281,0.0,2019-01-01,2026-06-30 23:00:00
se4_metered_losses,11.76281,0.0,2019-01-01,2026-06-30 23:00:00


# 02 ERA5-Land Reanalysis Weather Data

Use hourly **ERA5-Land reanalysis data** from the Copernicus Climate Data Store as weather features for model training. The dataset provides weather information at approximately 9 km spatial resolution (0.1° × 0.1° grid).

The data are extracted for **Stockholm (59.329° N, 18.068° E)** as a representative location for the SE3 bidding zone. Selected variables include temperature, dew-point temperature, wind components, precipitation, and solar radiation.

**Data source:**  
https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download

## Weather variables and units

All values and column names are kept in their original ERA5-Land units. No unit conversions are applied.

| Column | Meaning | Original unit |
|---|---|---|
| `t2m` | Air temperature at 2 m | Kelvin (K) |
| `d2m` | Dewpoint temperature at 2 m | K |
| `skt` | Surface skin temperature | K |
| `stl1`?`stl4` | Soil temperature, layers 1?4 | K |
| `u10`, `v10` | Eastward/northward wind components at 10 m | m/s |
| `sp` | Surface pressure | Pa |
| `tp` | Accumulated precipitation | m of water |
| `snowc` | Snow cover | % |
| `sde` | Snow depth | m |
| `swvl1`?`swvl4` | Volumetric soil water, layers 1?4 | m?/m? |
| `ssrd` | Downward solar radiation energy | J/m? |
| `strd` | Downward thermal radiation energy | J/m? |
| `latitude`, `longitude` | Grid coordinates | Degrees |

Precipitation and radiation are accumulated quantities. Verify the product's accumulation convention before interpreting them as hourly totals; hourly timestamps alone do not establish the accumulation interval.

Sources: [ERA5-Land time-series catalogue](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview) and [ECMWF ERA5-Land documentation](https://confluence.ecmwf.int/pages/viewpage.action?pageId=143043366).

In [9]:
# Output folder
output_dir = Path("data/weather/ECMWF")
output_dir.mkdir(parents=True, exist_ok=True)

# Output file
output_file = output_dir / "era5_land_stockholm_2018_2026.zip"

# ERA5-Land dataset
dataset = "reanalysis-era5-land-timeseries"

# Request configuration
request = {
    "variable": [
        "2m_dewpoint_temperature",
        "2m_temperature",
        "surface_pressure",
        "total_precipitation",
        "surface_solar_radiation_downwards",
        "surface_thermal_radiation_downwards",
        "skin_temperature",
        "snow_cover",
        "snow_depth",
        "soil_temperature_level_1",
        "soil_temperature_level_2",
        "soil_temperature_level_3",
        "soil_temperature_level_4",
        "volumetric_soil_water_level_1",
        "volumetric_soil_water_level_2",
        "volumetric_soil_water_level_3",
        "volumetric_soil_water_level_4",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
    ],
    "location": {
        "longitude": 18.068,
        "latitude": 59.329,
    },
    "date": ["2018-01-01/2026-09-01"],
    "data_format": "csv",
}

# Connect to the Copernicus Climate Data Store
client = cdsapi.Client()

# Download the data
client.retrieve(
    dataset,
    request,
    str(output_file),
)

print(f"Downloaded to: {output_file}")

2026-09-21 20:04:56,827 INFO Request ID is 566559cf-3e07-4ef3-a160-b52e71f43c77
2026-09-21 20:04:58,136 INFO status has been updated to accepted
2026-09-21 20:05:12,753 INFO status has been updated to running
2026-09-21 20:05:20,433 INFO status has been updated to successful
                                                                                          

Downloaded to: data\weather\ECMWF\era5_land_stockholm_2018_2026.zip


In [13]:
zip_file = Path("data/weather/ECMWF/era5_land_stockholm_2018_2026.zip")
output_dir = Path("data/clean/weather")
output_dir.mkdir(parents=True, exist_ok=True)

merged_file = output_dir / "era5_land_stockholm_2018_2026_merged.csv"

merge_keys = ["valid_time", "latitude", "longitude"]
frames = []

with ZipFile(zip_file) as archive:
    for name in sorted(archive.namelist()):
        if name.lower().endswith(".csv"):
            with archive.open(name) as file:
                frames.append(
                    pd.read_csv(file, parse_dates=["valid_time"])
                )

df_weather = reduce(
    lambda left, right: left.merge(
        right,
        on=merge_keys,
        how="outer",
        validate="one_to_one",
    ),
    frames,
).set_index("valid_time").sort_index()

df_weather.to_csv(merged_file)